# Datamine NSTRANS Process Example

This notebook demonstrates how to configure and run the **`nstrans`** process wrapper in `dmstudio`.

## Process Description

## NSTRANS

This process transforms a set of data to a normal distribution.

This process is run as part of Advanced Estimation if the Transform | Normal Score option is selected on the [Create Variograms](<../STUDIO_RM/Multivariate_Create_Variograms.md>) panel.

The input to this process is a sample data file containing at least one grade field to be transformed. It may also include a declustering weight field (*DCWGT). The file does not need XYZ coordinate data in order to do the transform. However coordinates will be required if variograms are to be calculated for the transformed data.

The process creates an output file containing transformed points. This will contain the same data as the input file, but with an extra transformed data field (*NSGRADE).

You can also restrict the scope of transformation by setting a minimum (@MINGRADE) and/or maximum (@MAXGRADE) grade value to transform.

**Note** : This process supports **[retrieval criteria](<../COMMON/Retrieval_Criteria_Overview.md>)**.

### Input Files:

* **in** (Samples file):
  Input sample data file. This must include the grade field **GRADE** and may also
  include the declustering weight field **DCWGT** .
  Required=Yes

### Output Files:

### Fields:

* **grade** (Numeric : IN):
  Field in the input sample file **IN** defining the grade to be transformed.
  Default=Undefined
  Required=Yes

* **dcwgt** (Numeric : IN):
  Optional declustering weights field in the **IN** file.
  Default=Undefined
  Required=No

* **nsgrade** (Alphanumeric : -):
  Field in the output file **OUT** containing the transformed grade.(This may be the same
  as **GRADE** , in which case it overwrites the original value)
  Default=Undefined
  Required=No

### Parameters:

* **mingrade**:
  Minimum **GRADE** value input from the IN sample file.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **maxgrade**:
  Maximum **GRADE** value input from the IN sample file.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('nstrans')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute nstrans
print("Running nstrans...")
dm_cmd.nstrans(
    in_i='t_assays',  # required
    grade_f='optional',  # required
    # dcwgt_f='optional',  # optional
    # nsgrade_f='optional',  # optional
    # mingrade_p=0,  # optional
    # maxgrade_p='optional',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("nstrans execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_nstrans_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")